# CrewAI + Neo4j: self-contained local MCP walkthrough

This notebook is the complete delivery: it defines its own local MCP adapter, read-only Neo4j tools, optional Neo4j Agent Memory Service (NAMS) tools, and CrewAI workflows. It does **not** import project modules or `custom_tools`.

Credentials stay outside this notebook. Load them from environment variables or a local `.env` file that is ignored by Git; never paste or print a credential here.


## Architecture and component mapping

| Component | Role in this notebook |
|---|---|
| **CrewAI** | Coordinates the single-agent question-answering crew and the sequential researcher/analyst/writer briefing crew. |
| **OpenAI model** | `gpt-5.4-mini` is the default model, overrideable with `OPENAI_MODEL_NAME`. |
| **Neo4j Python driver** | Performs parameterized, read-only Cypher operations directly against the graph. |
| **Local Neo4j MCP server** | Runs as a local stdio subprocess; its tools are discovered dynamically and converted into typed CrewAI tools. |
| **NAMS** | Optional long-term memory. Agents can explicitly recall preferences/facts and save verified findings for later sessions. |

The direct Neo4j tools are useful for predictable graph operations. MCP is useful when the local Neo4j MCP server exposes additional capabilities. NAMS is independent of the operational graph and is deliberately optional.


## 1. Install dependencies with `uv`

Run once in the notebook kernel. This installs packages into the kernel's interpreter; it does not create a repository dependency file. If your environment already has `uv`, the first command is harmless.


In [1]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install", "--system-certs", "--python", sys.executable,
        "crewai>=0.100.0", "neo4j>=5.18.0", "mcp>=1.0.0",
        "neo4j-mcp-server>=1.5.3", "neo4j-agent-memory>=0.2.0",
        "pydantic>=2.0.0", "python-dotenv>=1.0.0",
    ],
    check=True,
)


Using Python 3.12.13 environment at: /Users/karan_chellani/agent-integrations/neo4j-agent-integrations.worktrees/update-model-to-gpt54-mini-local-mcp/crewai/.venv
Checked 7 packages in 88ms


CompletedProcess(args=['/Users/karan_chellani/agent-integrations/neo4j-agent-integrations.worktrees/update-model-to-gpt54-mini-local-mcp/crewai/.venv/bin/python', '-m', 'uv', 'pip', 'install', '--system-certs', '--python', '/Users/karan_chellani/agent-integrations/neo4j-agent-integrations.worktrees/update-model-to-gpt54-mini-local-mcp/crewai/.venv/bin/python', 'crewai>=0.100.0', 'neo4j>=5.18.0', 'mcp>=1.0.0', 'neo4j-mcp-server>=1.5.3', 'neo4j-agent-memory>=0.2.0', 'pydantic>=2.0.0', 'python-dotenv>=1.0.0'], returncode=0)

## 2. Configure credentials externally

Create a local `.env` beside the notebook or export variables in the shell that launches Jupyter. Required variables are `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD`. `NEO4J_DATABASE` defaults to `neo4j`.

For local MCP, set `MCP_SERVER_COMMAND` (for example `neo4j-mcp-server`) and optionally its `NEO4J_MCP_*` settings. The notebook maps ordinary Neo4j settings to the MCP names without displaying either values or secrets. NAMS is enabled only when its package is installed and `MEMORY_API_KEY` is available.


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

# The current directory is intentionally the only implicit configuration location.
load_dotenv(Path.cwd() / ".env", override=False)

required = ("OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Set the required environment variables before continuing: " + ", ".join(missing))

DEFAULT_MODEL = os.environ.get("OPENAI_MODEL_NAME", "gpt-5.4-mini")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")
print(f"Configured model: {DEFAULT_MODEL}; database: {NEO4J_DATABASE}")


Configured model: gpt-5.4-mini; database: e2cf14df


## 3. Imports and safe synchronous bridge

Jupyter may already own an event loop. The small bridge below runs async MCP/NAMS calls on a separate thread in that case, so the same functions work in a notebook and in a standard synchronous CrewAI run.


In [3]:
import asyncio
import json
import re
import shlex
import shutil
import sysconfig
import threading
from functools import lru_cache
from pathlib import Path
from typing import Any

from crewai import Agent, Crew, Process, Task
from crewai.tools import BaseTool
from neo4j import GraphDatabase, RoutingControl
from pydantic import BaseModel, Field, create_model

try:
    from mcp import ClientSession
    from mcp.client.stdio import StdioServerParameters, stdio_client
    HAS_MCP = True
except ImportError:
    HAS_MCP = False

try:
    from neo4j_agent_memory import MemoryClient
    HAS_NAMS = True
except ImportError:
    HAS_NAMS = False

print(f"Core libraries loaded. Local MCP available: {HAS_MCP}; NAMS library available: {HAS_NAMS}.")


def run_sync(coro: Any) -> Any:
    """Run a coroutine from normal Python or an active notebook event loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}

    def runner() -> None:
        try:
            result["value"] = asyncio.run(coro)
        except BaseException as error:
            result["error"] = error

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


Core libraries loaded. Local MCP available: True; NAMS library available: True.


## 4. Direct read-only Neo4j operations

All values are passed as Cypher parameters. The raw-query helper rejects multiple statements and write/administration keywords before execution. The purpose-built operations below are examples that can be adapted to the labels and relationship types in your graph.


In [4]:
WRITE_OR_ADMIN_KEYWORDS = re.compile(
    r"\b(CREATE|MERGE|SET|DELETE|DETACH|REMOVE|DROP|ALTER|RENAME|GRANT|DENY|REVOKE|LOAD\s+CSV|FOREACH)\b",
    re.IGNORECASE,
)
READ_START = re.compile(r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH|UNWIND|CALL\s+db\.(index|schema)|RETURN|EXPLAIN|PROFILE)\b", re.IGNORECASE)


@lru_cache(maxsize=1)
def neo4j_driver() -> Any:
    return GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )


def read_cypher(query: str, parameters: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    """Execute one parameterized, non-mutating Cypher statement."""
    normalized = query.strip()
    if ";" in normalized.rstrip(";") or WRITE_OR_ADMIN_KEYWORDS.search(normalized):
        raise ValueError("Only a single read-only Cypher statement is allowed.")
    if not READ_START.match(normalized):
        raise ValueError("The query must begin with a read-only Cypher clause.")
    records, _, _ = neo4j_driver().execute_query(
        normalized,
        parameters_=parameters or {},
        database_=NEO4J_DATABASE,
        routing_=RoutingControl.READ,
    )
    return [record.data() for record in records]


def company_profile(company_name: str) -> list[dict[str, Any]]:
    return read_cypher(
        """
        MATCH (company:Organization)
        WHERE toLower(company.name) CONTAINS toLower($company_name)
        OPTIONAL MATCH (company)-[rel]-(connected)
        RETURN company.name AS company, company.summary AS summary,
               collect(DISTINCT labels(connected))[..10] AS connected_node_labels,
               collect(DISTINCT type(rel))[..15] AS relationship_types
        LIMIT 5
        """,
        {"company_name": company_name},
    )


def related_organizations(company_name: str, max_hops: int = 2) -> list[dict[str, Any]]:
    hops = max(1, min(int(max_hops), 3))
    return read_cypher(
        f"""
        MATCH path = (company:Organization)-[*1..{hops}]-(related:Organization)
        WHERE toLower(company.name) CONTAINS toLower($company_name) AND company <> related
        RETURN DISTINCT related.name AS organization,
               [rel IN relationships(path) | type(rel)] AS relationship_types,
               length(path) AS hops
        ORDER BY hops, organization
        LIMIT 25
        """,
        {"company_name": company_name},
    )


def graph_overview() -> list[dict[str, Any]]:
    return read_cypher(
        "MATCH (node) UNWIND labels(node) AS label RETURN label, count(*) AS node_count ORDER BY node_count DESC LIMIT 25"
    )

print("Read-only Neo4j helpers registered: graph overview, company profile, and relationship traversal.")


Read-only Neo4j helpers registered: graph overview, company profile, and relationship traversal.


## 5. Turn direct graph operations into individual CrewAI tools

These are the notebook's built-in tools. `ReadCypherTool` is intentionally read-only; give agents narrow tools first and use raw Cypher only when the question needs it.


In [5]:
class CompanyProfileInput(BaseModel):
    company_name: str = Field(..., description="Company name or distinctive fragment.")


class CompanyProfileTool(BaseTool):
    name: str = "company_profile"
    description: str = "Return a graph-backed company profile and its observed relationship types."
    args_schema: type[BaseModel] = CompanyProfileInput

    def _run(self, company_name: str) -> str:
        return json.dumps(company_profile(company_name), indent=2, default=str)


class RelatedOrganizationsInput(BaseModel):
    company_name: str = Field(..., description="Company name or distinctive fragment.")
    max_hops: int = Field(default=2, ge=1, le=3, description="Traversal depth from 1 through 3.")


class RelatedOrganizationsTool(BaseTool):
    name: str = "related_organizations"
    description: str = "Find organizations related to a company through up to three graph hops."
    args_schema: type[BaseModel] = RelatedOrganizationsInput

    def _run(self, company_name: str, max_hops: int = 2) -> str:
        return json.dumps(related_organizations(company_name, max_hops), indent=2, default=str)


class ReadCypherInput(BaseModel):
    query: str = Field(..., description="One read-only Cypher statement. Values must use parameters where possible.")
    parameters_json: str = Field(default="{}", description="JSON object containing Cypher parameters.")


class ReadCypherTool(BaseTool):
    name: str = "read_cypher"
    description: str = "Run a single parameterized, read-only Cypher query; writes and admin commands are rejected."
    args_schema: type[BaseModel] = ReadCypherInput

    def _run(self, query: str, parameters_json: str = "{}") -> str:
        try:
            parameters = json.loads(parameters_json)
            if not isinstance(parameters, dict):
                raise ValueError("parameters_json must contain a JSON object")
            return json.dumps(read_cypher(query, parameters), indent=2, default=str)
        except Exception as error:
            return json.dumps({"error": str(error)})


def direct_graph_tools() -> list[BaseTool]:
    return [CompanyProfileTool(), RelatedOrganizationsTool(), ReadCypherTool()]

print("Direct graph tools:", ", ".join(tool.name for tool in direct_graph_tools()))


Direct graph tools: company_profile, related_organizations, read_cypher


## 6. Discover and invoke local stdio MCP tools with typed inputs

The MCP server is never imported as a project module. At discovery time, each MCP JSON input schema becomes a Pydantic input model for a CrewAI `BaseTool`. The server process inherits environment settings but nothing in this notebook logs them.


In [6]:
def local_mcp_environment() -> dict[str, str]:
    environment = dict(os.environ)
    for mcp_name, neo4j_name in {
        "NEO4J_MCP_URI": "NEO4J_URI",
        "NEO4J_MCP_USERNAME": "NEO4J_USERNAME",
        "NEO4J_MCP_PASSWORD": "NEO4J_PASSWORD",
        "NEO4J_MCP_DATABASE": "NEO4J_DATABASE",
    }.items():
        if not environment.get(mcp_name) and environment.get(neo4j_name):
            environment[mcp_name] = environment[neo4j_name]
    environment.setdefault("NEO4J_MCP_READ_ONLY", "true")
    environment.setdefault("NEO4J_TELEMETRY", "false")
    return environment


def mcp_server_parameters(command: str) -> Any:
    if not HAS_MCP:
        raise RuntimeError("Install the 'mcp' package to use local MCP.")
    parts = shlex.split(command)
    if not parts:
        raise ValueError("MCP_SERVER_COMMAND must contain an executable command.")
    executable = parts[0]
    if not os.path.dirname(executable):
        executable = shutil.which(executable) or str(Path(sysconfig.get_path("scripts")) / executable)
    return StdioServerParameters(command=executable, args=parts[1:], env=local_mcp_environment())


async def discover_mcp_tools(command: str) -> list[dict[str, Any]]:
    async with (
        stdio_client(mcp_server_parameters(command)) as (read_stream, write_stream),
        ClientSession(read_stream, write_stream) as session,
    ):
        await session.initialize()
        response = await session.list_tools()
        return [
            {"name": tool.name, "description": tool.description or "", "input_schema": tool.inputSchema}
            for tool in response.tools
        ]


async def invoke_mcp_tool(command: str, tool_name: str, arguments: dict[str, Any]) -> str:
    async with (
        stdio_client(mcp_server_parameters(command)) as (read_stream, write_stream),
        ClientSession(read_stream, write_stream) as session,
    ):
        await session.initialize()
        response = await session.call_tool(tool_name, arguments=arguments)
        text = [item.text for item in response.content if hasattr(item, "text")]
        return "\n".join(text) if text else json.dumps(response.model_dump(), default=str)


def json_schema_type(schema: dict[str, Any]) -> type[Any]:
    return {"array": list[Any], "boolean": bool, "integer": int, "number": float,
            "object": dict[str, Any], "string": str}.get(schema.get("type"), Any)


def typed_mcp_input(tool_name: str, schema: dict[str, Any]) -> type[BaseModel]:
    properties = schema.get("properties", {})
    required = set(schema.get("required", []))
    fields = {
        name: (json_schema_type(field_schema), Field(
            default=... if name in required else field_schema.get("default"),
            description=field_schema.get("description", ""),
        ))
        for name, field_schema in properties.items()
    }
    model_name = "".join(piece.title() for piece in re.split(r"[-_]", tool_name))
    return create_model(f"{model_name}MCPInput", **fields)


class DynamicMCPTool(BaseTool):
    name: str = "mcp_tool"
    description: str = "Invoke a local MCP tool."
    server_command: str = ""
    target_tool_name: str = ""
    args_schema: type[BaseModel] = BaseModel

    def _run(self, **arguments: Any) -> str:
        return run_sync(invoke_mcp_tool(self.server_command, self.target_tool_name, arguments))


def load_local_mcp_tools(command: str | None = None) -> list[BaseTool]:
    command = command or os.environ.get("MCP_SERVER_COMMAND", "").strip()
    if not command:
        return []
    metadata = run_sync(discover_mcp_tools(command))
    tools = [
        DynamicMCPTool(
            name=f"mcp_{item['name']}",
            description=f"[Local MCP] {item['description'] or item['name']}",
            server_command=command,
            target_tool_name=item["name"],
            args_schema=typed_mcp_input(item["name"], item["input_schema"]),
        )
        for item in metadata
        if not item["name"].startswith("write-")
    ]
    return tools

print("Embedded CrewAI, local MCP, Neo4j, and optional NAMS dependencies are ready.")


Embedded CrewAI, local MCP, Neo4j, and optional NAMS dependencies are ready.


### Verify MCP discovery and a typed call

Set `MCP_SERVER_COMMAND` before running this cell. Discovery names and descriptions are safe to display. The optional call selects a schema tool if the server offers one; inspect `mcp_tools` to choose another tool and supply only its documented arguments.


In [7]:
mcp_tools = load_local_mcp_tools()
if not mcp_tools:
    print("MCP is not configured. Set MCP_SERVER_COMMAND to enable local stdio MCP tools.")
else:
    for tool in mcp_tools:
        print(f"- {tool.name}: {tool.description}")

    schema_tool = next((tool for tool in mcp_tools if "schema" in tool.name.lower()), None)
    if schema_tool:
        print("\nSchema response:\n", schema_tool.run())


- mcp_get-schema: [Local MCP] 
		Retrieve the schema information from the Neo4j database, including node labels, relationship types, and property keys.
		If the database contains no data, no schema information is returned.
- mcp_read-cypher: [Local MCP] read-cypher can run only read-only Cypher statements. For write operations (CREATE, MERGE, DELETE, SET, etc...), schema/admin commands, or PROFILE queries, use write-cypher instead.



Schema response:
 [{"key":"Concept","value":{"type":"node","properties":{"canonicalName":"STRING","confidence":"FLOAT","createdAt":"DATE_TIME","description":"STRING","embedding":"LIST","id":"STRING","mentionCount":"INTEGER","name":"STRING","nameEmbedding":"LIST","needsReview":"BOOLEAN","normName":"STRING","ontologyVersionId":"STRING","sourceStage":"STRING","systemAdded":"BOOLEAN","type":"STRING","updatedAt":"DATE_TIME","workspaceId":"STRING"},"relationships":{"EXTRACTED_FROM":{"direction":"out","labels":["Message"],"properties":{"confidence":"FLOAT","extractedAt":"DATE_TIME","method":"STRING","sourceStage":"STRING"}},"MENTIONS":{"direction":"in","labels":["Message"]},"RELATED_TO":{"direction":"out","labels":["Entity","Event"],"properties":{"confidence":"FLOAT","firstSeenAt":"DATE_TIME","lastSeenAt":"DATE_TIME","method":"STRING","predicate":"STRING","sourceMessages":"LIST"}}}}},{"key":"Entity","value":{"type":"node","properties":{"canonicalName":"STRING","confidence":"FLOAT","createdAt

## 7. Optional NAMS: explicit durable-memory use case

A useful memory pattern is preserving a **verified analytical takeaway** after a briefing and recalling it in a later briefing. The demonstration is disabled by default and writes only when you explicitly set `SAVE_NAMS_DEMO = True` after replacing the sample with a verified fact.


In [8]:
_memory_client: Any | None = None


def nams_enabled() -> bool:
    return HAS_NAMS and bool(os.environ.get("MEMORY_API_KEY"))


def memory_client() -> Any | None:
    global _memory_client
    if _memory_client is None and nams_enabled():
        _memory_client = MemoryClient()
    return _memory_client


async def search_memory(query: str, limit: int = 5) -> list[str]:
    client = memory_client()
    if client is None:
        return []
    findings: list[str] = []
    for entity in await client.long_term.search_entities(query, limit=limit):
        name = getattr(entity, "display_name", getattr(entity, "name", "Entity"))
        description = getattr(entity, "description", "")
        findings.append(f"{name}: {description}" if description else str(name))
    return findings[:limit]


async def save_memory_fact(subject: str, predicate: str, content: str) -> None:
    client = memory_client()
    if client is None:
        raise RuntimeError("NAMS is not configured; set MEMORY_API_KEY or compatible NAMS settings.")
    await client.long_term.add_fact(subject, predicate, content)


class SearchMemoryInput(BaseModel):
    query: str = Field(..., description="Topic to recall from long-term graph memory.")
    limit: int = Field(default=5, ge=1, le=10)


class SearchMemoryTool(BaseTool):
    name: str = "search_memory"
    description: str = "Recall previously saved, cross-session facts from NAMS long-term memory."
    args_schema: type[BaseModel] = SearchMemoryInput

    def _run(self, query: str, limit: int = 5) -> str:
        if not nams_enabled():
            return json.dumps({"status": "disabled", "message": "NAMS is not configured."})
        return json.dumps({"memories": run_sync(search_memory(query, limit))}, indent=2)


class SaveMemoryFactInput(BaseModel):
    subject: str = Field(..., description="Subject of a verified fact.")
    predicate: str = Field(..., description="Relationship describing the fact.")
    content: str = Field(..., description="Verified fact or analytical takeaway to retain.")


class SaveMemoryFactTool(BaseTool):
    name: str = "save_memory_fact"
    description: str = "Explicitly save a verified finding to NAMS for future crews and sessions."
    args_schema: type[BaseModel] = SaveMemoryFactInput

    def _run(self, subject: str, predicate: str, content: str) -> str:
        if not nams_enabled():
            return json.dumps({"status": "disabled", "message": "NAMS is not configured."})
        run_sync(save_memory_fact(subject, predicate, content))
        return json.dumps({"status": "saved", "subject": subject, "predicate": predicate})


def nams_tools() -> list[BaseTool]:
    return [SearchMemoryTool(), SaveMemoryFactTool()] if nams_enabled() else []

print("NAMS tools enabled." if nams_enabled() else "NAMS tools are disabled until MEMORY_API_KEY is configured.")


NAMS tools enabled.


In [9]:
# Set this to True only after replacing the sample values with a verified finding.
SAVE_NAMS_DEMO = False
if SAVE_NAMS_DEMO and nams_enabled():
    SaveMemoryFactTool().run(
        subject="Example Company",
        predicate="briefing_preference",
        content="Use concise Markdown with verified graph facts separated from analytical inferences.",
    )
    print(SearchMemoryTool().run(query="Example Company briefing preference"))
elif not nams_enabled():
    print("NAMS demo skipped: configure MEMORY_API_KEY to enable durable memory.")
else:
    print("NAMS write skipped. Set SAVE_NAMS_DEMO = True to save a verified finding.")


NAMS write skipped. Set SAVE_NAMS_DEMO = True to save a verified finding.


## 8. Natural-language graph question

The agent receives direct graph tools, dynamically discovered MCP tools when configured, and NAMS tools when configured. It must use tool results for factual claims and clearly distinguish graph evidence from inference.

**Example questions**

- “What relationship types connect Acme Corp to other organizations within two hops?”
- “Find organizations related to a company and identify the shortest paths.”
- “Which labels are most common in this graph?”
- “What briefing format preferences have we saved for this company?”


In [10]:
def available_tools() -> list[BaseTool]:
    return direct_graph_tools() + load_local_mcp_tools() + nams_tools()


def build_graph_query_crew(question: str) -> Crew:
    agent = Agent(
        role="Neo4j Graph Intelligence Analyst",
        goal="Answer graph questions with verified evidence from the available read-only tools.",
        backstory=(
            "You are precise about graph evidence. Use a tool before making factual claims, "
            "state when data is missing, and distinguish results from inference."
        ),
        tools=available_tools(),
        llm=DEFAULT_MODEL,
        verbose=True,
        allow_delegation=False,
    )
    task = Task(
        description=(
            "Answer this question using the graph tools. Use only verified tool output for factual claims. "
            f"Question: {question}"
        ),
        expected_output="A concise Markdown answer that cites the retrieved graph evidence in prose.",
        agent=agent,
    )
    return Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=True)


question = "Show all entities connected to DB and their relationship types."
result = run_sync(asyncio.to_thread(build_graph_query_crew(question).kickoff))
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: adfa7cfd-0471-4090-be0e-b4456f7dcf7e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question using the graph tools. Use only verified tool output for factual claims. Question:  │
│  Show all entities connected to DB and their relationship types.                                                │
│  ID: ca1176f5-40cb-4b01-8aa9-786c4691be43                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Neo4j Graph Intelligence Analyst                                                                        │
│                                                                                                                 │
│  Task: Answer this question using the graph tools. Use only verified tool output for factual claims. Question:  │
│  Show all entities connected to DB and their relationship types.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: company_profile                                                                                          │
│  Args: {'company_name': 'DB'}                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:neo4j.pool:Unable to retrieve routing information


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Neo4j Graph Intelligence Analyst                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I couldn’t verify any connected entities for **DB** from the graph tools because the available read/query      │
│  operations returned routing errors.                                                                            │
│                                                                                                                 │
│  ### Verified evidence                                                                                          │
│  - The schema confirms the graph contains node labels such as **Database**, **Entity**, **Organization**,       │
│  **Concept**, **Object**, and **Event**, and relationship types including **RELATED_TO**, **SAME_AS**,          │
│  **EXTRACTED_FROM**, **MENTIONS**, **PARTICIPATED_IN**, and **RELATES_TO**.                                     │
│  - However, the actual graph query intended to find **DB** and list its connected entities failed with:         │
│  **“Unable to retrieve routing information.”**                                                                  │
│                                                                                                                 │
│  ### Result                                                                                                     │
│  - **Connected entities to DB:** not retrievable from the current tool session.                                 │
│  - **Relationship types for DB:** not verified due to query failure.                                            │
│                                                                                                                 │
│  If you want, I can try a narrower lookup next (for example, searching only the **Database** label or only      │
│  exact name matches) once the routing issue is available to the tools.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question using the graph tools. Use only verified tool output for factual claims. Question:  │
│  Show all entities connected to DB and their relationship types.                                                │
│  Agent: Neo4j Graph Intelligence Analyst                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I couldn’t verify any connected entities for **DB** from the graph tools because the available read/query operations returned routing errors.

### Verified evidence
- The schema confirms the graph contains node labels such as **Database**, **Entity**, **Organization**, **Concept**, **Object**, and **Event**, and relationship types including **RELATED_TO**, **SAME_AS**, **EXTRACTED_FROM**, **MENTIONS**, **PARTICIPATED_IN**, and **RELATES_TO**.
- However, the actual graph query intended to find **DB** and list its connected entities failed with: **“Unable to retrieve routing information.”**

### Result
- **Connected entities to DB:** not retrievable from the current tool session.
- **Relationship types for DB:** not verified due to query failure.

If you want, I can try a narrower lookup next (for example, searching only the **Database** label or only exact name matches) once the routing issue is available to the tools.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: adfa7cfd-0471-4090-be0e-b4456f7dcf7e                                                                       │
│  Final Output: I couldn’t verify any connected entities for **DB** from the graph tools because the available   │
│  read/query operations returned routing errors.                                                                 │
│                                                                                                                 │
│  ### Verified evidence                                                                                          │
│  - The schema confirms the graph contains node labels such as **Database**, **Entity**, **Organization**,       │
│  **Concept**, **Object**, and **Event**, and relationship types including **RELATED_TO**, **SAME_AS**,          │
│  **EXTRACTED_FROM**, **MENTIONS**, **PARTICIPATED_IN**, and **RELATES_TO**.                                     │
│  - However, the actual graph query intended to find **DB** and list its connected entities failed with:         │
│  **“Unable to retrieve routing information.”**                                                                  │
│                                                                                                                 │
│  ### Result                                                                                                     │
│  - **Connected entities to DB:** not retrievable from the current tool session.                                 │
│  - **Relationship types for DB:** not verified due to query failure.                                            │
│                                                                                                                 │
│  If you want, I can try a narrower lookup next (for example, searching only the **Database** label or only      │
│  exact name matches) once the routing issue is available to the tools.                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 9. Multi-agent company briefing

The crew runs three focused agents sequentially: a researcher gathers company facts, an analyst explores network structure, and a writer produces an executive briefing. The writer may use NAMS only when it is configured; it should save only verified conclusions that are useful across sessions.


In [11]:
def build_company_briefing_crew(company_name: str) -> Crew:
    graph_tools = direct_graph_tools() + load_local_mcp_tools()
    memory_tools = nams_tools()

    researcher = Agent(
        role="Knowledge Graph Researcher",
        goal="Retrieve verified company facts, graph labels, and relationship evidence.",
        backstory="You ground every finding in read-only graph tool results.",
        tools=graph_tools + memory_tools,
        llm=DEFAULT_MODEL,
        verbose=True,
        allow_delegation=False,
    )
    analyst = Agent(
        role="Graph Network Analyst",
        goal="Analyze relationship paths, clusters, dependencies, and uncertainty from graph evidence.",
        backstory="You identify non-obvious structure without inventing missing links.",
        tools=graph_tools + memory_tools,
        llm=DEFAULT_MODEL,
        verbose=True,
        allow_delegation=False,
    )
    writer = Agent(
        role="Executive Briefing Writer",
        goal="Produce a crisp, decision-ready Markdown briefing grounded in prior task results.",
        backstory="You separate verified facts from implications and apply recalled preferences when available.",
        tools=memory_tools,
        llm=DEFAULT_MODEL,
        verbose=True,
        allow_delegation=False,
    )

    research = Task(
        description=(
            f"Research '{company_name}' in Neo4j. Retrieve a profile and relevant graph connections. "
            "Return only verified findings and note absent data."
        ),
        expected_output="Markdown research notes with company facts and supporting graph observations.",
        agent=researcher,
    )
    analysis = Task(
        description=(
            f"Analyze the relationship network for '{company_name}' using up to two graph hops. "
            "Identify material connections, possible dependencies, and uncertainty."
        ),
        expected_output="Markdown network analysis grounded in retrieved paths.",
        agent=analyst,
        context=[research],
    )
    briefing = Task(
        description=(
            f"Write an executive briefing for '{company_name}' using the research and analysis. "
            "Use sections: Executive Summary, Verified Graph Facts, Network Analysis, Strategic Implications, "
            "and Open Questions. Clearly label inferences. If NAMS is available, recall relevant preferences; "
            "save a fact only if it is verified and useful across future sessions."
        ),
        expected_output="A decision-ready Markdown company briefing with facts separated from inferences.",
        agent=writer,
        context=[research, analysis],
    )
    return Crew(
        agents=[researcher, analyst, writer],
        tasks=[research, analysis, briefing],
        process=Process.sequential,
        verbose=True,
    )


company_name = "Google"
briefing = run_sync(asyncio.to_thread(build_company_briefing_crew(company_name).kickoff))
print(briefing)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 43eff2cb-46c5-450a-9ff1-327d18ca9e19                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research 'Google' in Neo4j. Retrieve a profile and relevant graph connections. Return only verified      │
│  findings and note absent data.                                                                                 │
│  ID: 2f7a408a-09fd-4e43-b9d0-5f30996dd836                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Knowledge Graph Researcher                                                                              │
│                                                                                                                 │
│  Task: Research 'Google' in Neo4j. Retrieve a profile and relevant graph connections. Return only verified      │
│  findings and note absent data.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: company_profile                                                                                          │
│  Args: {'company_name': 'Google'}                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 🔧 Tool Error (#1) ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: search_memory                                                                                            │
│  Iteration: 1                                                                                                   │
│  Attempt: 0                                                                                                     │
│  Error: Client not connected. Use 'async with' or call connect().                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Briefing Writer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Google — Executive Briefing                                                                                  │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  **Verified fact:** The current Neo4j snapshot contains **no retrievable Google node**. Searches over           │
│  `Organization` nodes and then across **all nodes** by `name` / `canonicalName` returned no match for           │
│  `"google"`.                                                                                                    │
│                                                                                                                 │
│  **Verified fact:** Because no Google record was found, **no graph connections, dependencies, or paths** could  │
│  be grounded in the data.                                                                                       │
│                                                                                                                 │
│  **Inference:** This dataset appears either incomplete, domain-limited, or not populated with Google-related    │
│  entities in the accessible extract. That limits any company-level network analysis.                            │
│                                                                                                                 │
│  **Decision implication:** There is **insufficient graph evidence** to brief Google as an entity in this        │
│  snapshot. Any strategic characterization would be speculative.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Verified Graph Facts                                                                                        │
│  - **No Google node was found** in the current Neo4j data.                                                      │
│  - Case-insensitive searches were run over:                                                                     │
│    - `Organization` nodes                                                                                       │
│    - all nodes by `name`                                                                                        │
│    - all nodes by `canonicalName`                                                                               │
│  - **No rows were returned** for Google in either pass.                                                         │
│  - The graph schema is present and includes these relevant labels:                                              │
│    - `Organization`                                                                                             │
│    - `Entity`                                                                                                   │
│    - `Concept`                                                                                                  │
│    - `Event`                                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an executive briefing for 'Google' using the research and analysis. Use sections: Executive        │
│  Summary, Verified Graph Facts, Network Analysis, Strategic Implications, and Open Questions. Clearly label     │
│  inferences. If NAMS is available, recall relevant preferences; save a fact only if it is verified and useful   │
│  across future sessions.                                                                                        │
│  Agent: Executive Briefing Writer                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Google — Executive Briefing

## Executive Summary
**Verified fact:** The current Neo4j snapshot contains **no retrievable Google node**. Searches over `Organization` nodes and then across **all nodes** by `name` / `canonicalName` returned no match for `"google"`.

**Verified fact:** Because no Google record was found, **no graph connections, dependencies, or paths** could be grounded in the data.

**Inference:** This dataset appears either incomplete, domain-limited, or not populated with Google-related entities in the accessible extract. That limits any company-level network analysis.

**Decision implication:** There is **insufficient graph evidence** to brief Google as an entity in this snapshot. Any strategic characterization would be speculative.

---

## Verified Graph Facts
- **No Google node was found** in the current Neo4j data.
- Case-insensitive searches were run over:
  - `Organization` nodes
  - all nodes by `name`
  - all nodes by `canonicalName`
- **No rows were returne

## Operational notes

- Close the driver when finished in a long-lived kernel: `neo4j_driver().close()`.
- Keep `.env` outside version control. This notebook intentionally neither reads secret values into output nor writes configuration files.
- For a new graph, begin with a small read-only schema/label query or the MCP schema tool, then adapt the example labels (`Organization`) and relationship patterns.
- If the local MCP server provides a tool that can mutate data, do not add it to a production crew unless it is separately reviewed and constrained. The direct tools in this notebook are read-only by design.
